In [10]:
from pathlib import Path
import ast
import json
import pandas as pd
from IPython.core.inputtransformer2 import TransformerManager

workspace = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path/'env/zshooter-stack.toml').exists())
notebooks = sorted(path for directory in ('notebooks', 'dev_notebooks', 'validation/xshooter') for path in (workspace/directory).glob('*.ipynb'))
transform = TransformerManager()
rows, inventory, logged_errors = [], [], []
for path in notebooks:
    notebook = json.loads(path.read_text())
    code_cells = syntax_errors = saved_exceptions = 0
    for cell in notebook['cells']:
        if cell['cell_type'] != 'code':
            continue
        code_cells += 1
        source = ''.join(cell['source'])
        try:
            ast.parse(transform.transform_cell(source), filename=f"{path.name}:{cell['id']}")
        except SyntaxError as error:
            syntax_errors += 1
            rows.append((str(path.relative_to(workspace)), cell['id'], 'SyntaxError', str(error)))
        for output in cell['outputs']:
            if output['output_type'] == 'error':
                saved_exceptions += 1
                rows.append((str(path.relative_to(workspace)), cell['id'], output['ename'], output['evalue']))
            if output['output_type'] == 'stream':
                for line in ''.join(output['text']).splitlines():
                    if 'Traceback (most recent call last)' in line or ' - ERROR:' in line:
                        logged_errors.append((str(path.relative_to(workspace)), cell['id'], line))
    inventory.append((str(path.relative_to(workspace)), code_cells, syntax_errors, saved_exceptions))
errors = pd.DataFrame(rows, columns=['notebook', 'cell', 'exception', 'message'])
inventory = pd.DataFrame(inventory, columns=['notebook', 'code_cells', 'syntax_errors', 'saved_exceptions'])
logged_errors = pd.DataFrame(logged_errors, columns=['notebook', 'cell', 'message'])
inventory.to_csv(workspace/'validation/xshooter/data/runtime/notebook_checks.csv', index=False)
errors.to_csv(workspace/'validation/xshooter/data/runtime/remaining_errors.csv', index=False)
logged_errors.to_csv(workspace/'validation/xshooter/data/runtime/logged_errors.csv', index=False)
print(f'Inspected {len(notebooks)} notebooks; source syntax, saved exceptions and logged tracebacks/errors only.')
display(inventory)
display(errors)
display(logged_errors)

py.warnings - WARNING: 02_Stellar.ipynb:de4b4c3f17feed3b:7: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.

py.warnings - WARNING: 02_Stellar.ipynb:de4b4c3f17feed3b:9: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.

py.warnings - WARNING: 02_Stellar.ipynb:dc350e9dd5b4618d:7: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.

py.warnings - WARNING: 02_Stellar.ipynb:dc350e9dd5b4618d:9: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.

py.warnings - WARNING: 02_Stellar.ipynb:6eee2198e4c9523:14: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean 

,notebook,code_cells,syntax_errors,saved_exceptions
0,dev_notebooks/playground_zshooter.ipynb,16,0,0
1,dev_notebooks/psf_adc.ipynb,19,0,0
2,dev_notebooks/sims_for_drp.ipynb,12,0,0
3,dev_notebooks/test_effect.ipynb,18,0,0
4,dev_notebooks/zshooter_science_validation.ipynb,43,0,0
5,notebooks/01_Kilonovae.ipynb,28,0,0
6,notebooks/01_Kilonovae_XSHOOTER.ipynb,17,0,0
7,notebooks/02_Flat_AB_Comparison.ipynb,12,0,0
8,notebooks/02_Stellar.ipynb,21,0,0
9,notebooks/03_Slit_Illumination_Lamps.ipynb,4,0,0


,notebook,cell,exception,message


,notebook,cell,message


# Saved-frame runtime checks

The initial audit of 22 tracked notebooks found five saved exceptions. The current audit above includes this validation notebook as well. See [the runtime assessment](../../docs/notebook-runtime-checks.md) for the fixes and the exact execution scope.

The reductions below require the saved frames from both kilonova notebooks. They reuse the actual stochastic images, standard counts and measured calibration files. The two small archived swaths retain the exact inputs that exposed the BLUE and UVB failures. Raising the native limit repairs BLUE; UVB instead needs the same-equation continuation in PyReduce's uncertainty refinement. No random realization is replaced and no clipping or convergence threshold is relaxed.

The faint-mode X-shooter NIR flux calibration still fails its continuum control. Successful runtime checks do not license its science fluxes; the primary notebook and report retain the failure.

Final saved-file audit: all 23 notebooks (the original 22 plus this check) have zero syntax errors, saved exceptions and logged tracebacks/errors. Existing invalid-escape deprecation warnings are visible in the parser output.

In [2]:
import logging
from time import perf_counter
import numpy as np
from astropy.io import fits
from pyreduce.spectra import Spectra
from scopesim.source.source_templates import ab_spectrum
from synphot.units import convert_flux
from zsdrp.ZSHOOTER import ZSHOOTER
import zsdrp.pipe

# Reuse the actual stochastic frames and their measured calibrations.
scene_dir = workspace/'notebooks/01_Kilonovae_detectors/SPEC/kilonova_X-1.3_NS-0.7_dark_slit-0.7_ao-0_optimal_variance'
reduced_dir = scene_dir/'reduced'
instrument = ZSHOOTER()
reference = ab_spectrum(mag=16)
reference_wave = reference.waveset
reference_flux = convert_flux(reference_wave, reference(reference_wave), 'FLAM')
steps = ['bias', 'flat', 'trace', 'wavecal_master', 'science']
logging.getLogger('pyreduce.trace_model').setLevel(logging.ERROR)
files = {}
for channel in instrument.config.channels:
    groups = instrument.sort_files(str(scene_dir), target='', night='', channel=channel, steps=steps)
    (_, group), = [entry for entry in groups if len(entry[1]['science'])]
    files[channel] = {name: group[name] for name in steps}
    if channel != 'YJ':
        files[channel]['trace'] = files[channel]['flat']
reduction_times = []
print({channel: len(group['science']) for channel, group in files.items()})

py.warnings - WARNING: /Users/jibailey/src/PyReduce-xshooter/pyreduce/combine_frames.py:15: DeprecationWarning: Please import `median_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import median_filter

py.warnings - WARNING: /Users/jibailey/src/PyReduce-xshooter/pyreduce/util.py:17: DeprecationWarning: Please import `median_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import median_filter

py.warnings - WARNING: /Users/jibailey/src/PyReduce-xshooter/pyreduce/trace.py:29: DeprecationWarning: Please import `gaussian_filter1d` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter1d, median_filter, uniform_filter1d

py.warnings - WARNING: /Use

100%|██████████| 114/114 [00:00<00:00, 1500.43it/s]

{'BLUE': 5, 'GREEN': 5, 'RED': 5, 'YJ': 5, 'H': 5, 'K': 5}


In [3]:
for phase in (20.,):
    object_name = f'KN_{phase}D_SUB'
    for channel, group in files.items():
        science_file, = [path for path in group['science'] if fits.getheader(path)['OBJECT'] == object_name]
        selected = {channel: {**group, 'science': [science_file]}}
        start = perf_counter()
        reduced = zsdrp.pipe.run_reduction(selected, channels=[channel], output_dir=reduced_dir, disable_steps=['mask'], print_params=False, disable_all_plots=True, reuse_cals=True, require_variance=True)
        reduced[channel]['spectra']['STANDARD'] = Spectra.read(reduced_dir/f'standard_{channel}_extracted_electrons.fits', raw=True)
        calibrated = zsdrp.pipe.run_flux_calibration_and_stitch(reduced, reference_wave=reference_wave, reference_flux=reference_flux, plot=False, output_dir=str(reduced_dir))
        reduction_times.append((phase, channel, perf_counter()-start))
        print(f'{object_name}, {channel}: {reduction_times[-1][2]:.1f} s', flush=True)
        del reduced, calibrated
reduction_table = pd.DataFrame(reduction_times, columns=['phase_day', 'arm', 'seconds'])
reduction_table.to_csv(workspace/'validation/xshooter/data/runtime/zshooter_20day_reduction.csv', index=False)
display(reduction_table)

Running reduction for channel BLUE...
zsdrp.steps - Processing KN_20.0D_SUB for science extraction with 1 images.
INFO - Using optimal extraction to produce spectrum
pyreduce.extract - Using optimal extraction to produce spectrum
INFO - Extraction backend: c
pyreduce.extract - Extraction backend: c
zsdrp.pipe - Performing blaze normalization for channel BLUE.
INFO - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/01_Kilonovae_detectors/SPEC/kilonova_X-1.3_NS-0.7_dark_slit-0.7_ao-0_optimal_variance/reduced/spectra_KN_20.0D_SUB_BLUE_fluxcal.fits
pyreduce.spectra - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/01_Kilonovae_detectors/SPEC/kilonova_X-1.3_NS-0.7_dark_slit-0.7_ao-0_optimal_variance/reduced/spectra_KN_20.0D_SUB_BLUE_fluxcal.fits
INFO - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/01_Kilonovae_detectors/SPEC/kilonova_X-1.3_NS-0.7_dark_slit-0.7_ao-0_optimal_variance/reduced/spectra_STANDARD_BLUE_blazecal.fits
pyreduce.spectra - Save

,phase_day,arm,seconds
0,20.0,BLUE,45.141014
1,20.0,GREEN,49.467567
2,20.0,RED,29.658505
3,20.0,YJ,30.628397
4,20.0,H,15.684415
5,20.0,K,10.271984


In [4]:
import scopesim as sim
from zs_scopesim_tools.helpers import configure_irdb_path
from zsdrp.XSHOOTER import XSHOOTER

configure_irdb_path()
xs_root = workspace/'notebooks/xshooter_kilonova_detectors'
xs_settings = {
    'faint': {'!INST.uvb_curr_slit': '1.0', '!INST.vis_curr_slit': '0.9', '!INST.nir_curr_slit': '0.9', '!DET.biny_uvb': 2, '!DET.biny_vis': 2},
    'high': {'!INST.uvb_curr_slit': '0.8', '!INST.vis_curr_slit': '0.9', '!INST.nir_curr_slit': '0.6', '!DET.biny_uvb': 1, '!DET.biny_vis': 1},
}
xs_trains = {mode: sim.OpticalTrain(sim.UserCommands(use_instrument='XSHOOTER', set_modes=['SPEC'], properties=settings, ignore_effects=['skycalc_atmosphere'])) for mode, settings in xs_settings.items()}
xs_files = {}
for mode in xs_settings:
    xs_files[mode] = {}
    for arm in ('UVB', 'VIS', 'NIR'):
        directory = xs_root/mode
        cals = directory/'calibrations'
        xs_files[mode][arm] = {'bias': [cals/f'bias_{arm}.fits'] if arm != 'NIR' else [], 'flat': [cals/f'flat_{arm}.fits'], 'trace': [cals/f'trace_{arm}.fits'], 'wavecal_master': [cals/f'arc_{arm}.fits'], 'science': [directory/'prepared'/f'kilonova_20d_{arm}.fits']}
print({mode: {arm: group['science'][0].exists() for arm, group in arms.items()} for mode, arms in xs_files.items()})

astar.scopesim.utils - Resolving time: 60310.25 assuming MJD format and UTC scale
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

astar.scopesim.utils - Resolving time: 60310.25 assuming MJD format and UTC scale
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

{'faint': {'UVB': True, 'VIS': True, 'NIR': True}, 'high': {'UVB': True, 'VIS': True, 'NIR': True}}


In [8]:
from pyreduce import cwrappers
from pyreduce.extraction_uncertainty import slit_response

convergence_rows = []
for filename in ('zshooter_blue20_swath.npz', 'xshooter_uvb20_swath.npz'):
    with np.load(workspace/'validation/xshooter/data/runtime'/filename) as archive:
        inputs = dict(archive)
    for name in ('osample', 'lambda_sf', 'lambda_sp', 'preset'):
        inputs[name] = inputs[name].item()
    previous = None
    for limit in (4000, 8000, 16000):
        start = perf_counter()
        native = cwrappers.slitdec(inputs['image'], np.ones_like(inputs['image']), inputs['good'].astype(np.uint8), inputs['ycen'], inputs['slitcurve'], inputs['slitdeltas'], osample=inputs['osample'], lambda_sL=inputs['lambda_sf'], lambda_sP=inputs['lambda_sp'], maxiter=limit, kappa=0)
        refined = slit_response(**(inputs | {'slit': native['slitfunction']}))
        if previous is not None:
            np.testing.assert_allclose(refined[0], previous[0], rtol=1e-9, atol=1e-9)
            np.testing.assert_allclose(refined[2], previous[2], rtol=1e-8, atol=1e-9)
        previous = refined
        convergence_rows.append((filename, limit, native['info'][3], perf_counter()-start, np.sum(np.abs(refined[1]))/np.abs(refined[1].sum())))
        print(convergence_rows[-1], flush=True)
convergence = pd.DataFrame(convergence_rows, columns=['swath', 'iteration_limit', 'native_iterations', 'seconds', 'absolute_to_signed_profile_integral'])
convergence.to_csv(workspace/'validation/xshooter/data/runtime/faint_swath_convergence.csv', index=False)
display(convergence)

('zshooter_blue20_swath.npz', 4000, np.float64(3304.0), 1.0184078750899062, np.float64(10.561858915542443))
('zshooter_blue20_swath.npz', 8000, np.float64(3304.0), 1.023695708019659, np.float64(10.561858915542443))
('zshooter_blue20_swath.npz', 16000, np.float64(3304.0), 1.0096510000294074, np.float64(10.561858915542443))
('xshooter_uvb20_swath.npz', 4000, np.float64(967.0), 12.97202716593165, np.float64(2.491586998601756))
('xshooter_uvb20_swath.npz', 8000, np.float64(967.0), 13.220613374956883, np.float64(2.491586998601756))
('xshooter_uvb20_swath.npz', 16000, np.float64(967.0), 13.342892708955333, np.float64(2.491586998601756))


,swath,iteration_limit,native_iterations,seconds,absolute_to_signed_profile_integral
0,zshooter_blue20_swath.npz,4000,3304.0,1.018408,10.561859
1,zshooter_blue20_swath.npz,8000,3304.0,1.023696,10.561859
2,zshooter_blue20_swath.npz,16000,3304.0,1.009651,10.561859
3,xshooter_uvb20_swath.npz,4000,967.0,12.972027,2.491587
4,xshooter_uvb20_swath.npz,8000,967.0,13.220613,2.491587
5,xshooter_uvb20_swath.npz,16000,967.0,13.342893,2.491587


In [7]:
xs_reduction_times = []
for mode, arms in xs_files.items():
    for arm, group in arms.items():
        armdir = xs_root/mode/'reduced'/arm
        start = perf_counter()
        reduced = zsdrp.pipe.run_reduction({arm: group}, channels=[arm], output_dir=armdir, instrument=XSHOOTER(), trace_list=xs_trains[mode]['spectral_traces'], require_variance=True, reuse_cals=True, disable_steps=['mask'], print_params=False, disable_all_plots=True)
        reduced[arm]['spectra']['STANDARD'] = Spectra.read(armdir/'standard_blazecal_counts.fits', raw=True)
        calibrated = zsdrp.pipe.run_flux_calibration_and_stitch(reduced, reference_wave=reference_wave, reference_flux=reference_flux, plot=False, output_dir=str(armdir))
        xs_reduction_times.append((mode, arm, perf_counter()-start))
        print(f'{mode}, {arm}: {xs_reduction_times[-1][2]:.1f} s', flush=True)
        del reduced, calibrated
xs_reduction_table = pd.DataFrame(xs_reduction_times, columns=['mode', 'arm', 'seconds'])
xs_reduction_table.to_csv(workspace/'validation/xshooter/data/runtime/xshooter_20day_reduction.csv', index=False)
display(xs_reduction_table)

Running reduction for channel UVB...
zsdrp.steps - Processing KILONOVA_20D for science extraction with 1 images.
INFO - Using optimal extraction to produce spectrum
pyreduce.extract - Using optimal extraction to produce spectrum
zsdrp.pipe - Performing blaze normalization for channel UVB.
INFO - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/xshooter_kilonova_detectors/faint/reduced/UVB/spectra_KILONOVA_20D_UVB_fluxcal.fits
pyreduce.spectra - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/xshooter_kilonova_detectors/faint/reduced/UVB/spectra_KILONOVA_20D_UVB_fluxcal.fits
INFO - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/xshooter_kilonova_detectors/faint/reduced/UVB/spectra_STANDARD_UVB_blazecal.fits
pyreduce.spectra - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/xshooter_kilonova_detectors/faint/reduced/UVB/spectra_STANDARD_UVB_blazecal.fits
INFO - Saved 1 spectra to: /Users/jibailey/src/zs-scopesim/notebooks/xshooter_ki

,mode,arm,seconds
0,faint,UVB,27.285004
1,faint,VIS,26.198861
2,faint,NIR,24.680006
3,high,UVB,35.002656
4,high,VIS,61.212706
5,high,NIR,23.450857


In [9]:
products = []
for phase in (1.5, 7.5, 20.):
    for arm in instrument.config.channels:
        products.append(('ZShooter', 'SPEC', phase, arm, reduced_dir/f'spectra_KN_{phase}D_SUB_{arm}_fluxcal.fits'))
    products.append(('ZShooter', 'SPEC', phase, 'stitched', reduced_dir/f'spectra_KN_{phase}D_SUB_stitched.fits'))
    for mode in xs_settings:
        for arm in ('UVB', 'VIS', 'NIR'):
            products.append(('X-shooter', mode, phase, arm, xs_root/mode/'reduced'/arm/f'spectra_KILONOVA_{phase:g}D_{arm}_fluxcal.fits'))
        products.append(('X-shooter', mode, phase, 'stitched', xs_root/mode/'reduced'/f'KILONOVA_{phase:g}D_stitched.fits'))
product_rows = []
for instrument_name, mode, phase, arm, path in products:
    data = Spectra.read(path, raw=True)
    retained = samples = 0
    for spectrum in data.data:
        valid = ~spectrum.mask
        assert valid.any(), path
        assert np.all(np.isfinite(spectrum.wave[valid])), path
        assert np.all(np.isfinite(spectrum.spec[valid])), path
        assert np.all(np.isfinite(spectrum.sig[valid])), path
        assert np.all(spectrum.sig[valid] >= 0), path
        assert spectrum.covariance.shape == (spectrum.spec.size, spectrum.spec.size), path
        assert np.all(np.isfinite(spectrum.covariance.data)), path
        # Both quantities are saved in the same flux units; only roundoff can differ.
        np.testing.assert_allclose(spectrum.covariance.diagonal()[valid], spectrum.sig[valid]**2, rtol=1e-10, atol=0)
        samples += spectrum.spec.size
        retained += valid.sum()
    product_rows.append((instrument_name, mode, phase, arm, samples, retained))
    del data
product_checks = pd.DataFrame(product_rows, columns=['instrument', 'mode', 'phase_day', 'arm', 'samples', 'retained_samples'])
product_checks.to_csv(workspace/'validation/xshooter/data/runtime/science_product_checks.csv', index=False)
print(f'{len(product_checks)} products have finite retained samples and consistent covariance diagonals.')
display(product_checks)

45 products have finite retained samples and consistent covariance diagonals.


,instrument,mode,phase_day,arm,samples,retained_samples
0,ZShooter,SPEC,1.5,BLUE,27175,27175
1,ZShooter,SPEC,1.5,GREEN,34640,34640
2,ZShooter,SPEC,1.5,RED,33617,33617
3,ZShooter,SPEC,1.5,YJ,25821,25821
4,ZShooter,SPEC,1.5,H,17329,17329
5,ZShooter,SPEC,1.5,K,16319,16319
6,ZShooter,SPEC,1.5,stitched,149932,149932
7,X-shooter,faint,1.5,UVB,10980,10980
8,X-shooter,faint,1.5,VIS,17753,17753
9,X-shooter,faint,1.5,NIR,20373,20373
